# 79. Word Search
**Difficulty:** 🟡 Medium · **Topic:** Matrix · **LeetCode:** https://leetcode.com/problems/word-search/

## 💡 Concepts

**Core concept(s):** **Backtracking DFS** across the grid, matching the word letter by letter.

**Why it applies here:** A word is a path of adjacent cells. From every cell that matches the first letter, DFS to neighbors matching the next letter, marking cells used so a path can't reuse one; undo the mark when you back out.

**Key intuition:** Follow the word cell to cell; mark a cell used while exploring, then restore it.

---

### 📚 Working with a Matrix (Grid)
A matrix is a list of rows. Common moves: walk with `(r, c)` coordinates, **transpose** (swap rows/columns), or use the first row/column as scratch space to save memory.

### 📚 What is Backtracking?
**Backtracking** tries a step, explores, then **undoes** it. On a grid: mark a cell used, search neighbors, then restore it.

---

**Prerequisite knowledge:**
- Grid DFS.
- Backtracking (mark / restore).

## 📝 Problem

Does the word exist in the grid as a path of adjacent (up/down/left/right) cells, no cell reused?

**Example**
```
board=[["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], word="ABCCED" -> True
```

> One approach: backtracking DFS.

### Approach — Backtracking DFS

**Idea:** Try each starting cell. DFS matches `word[i]`, marks the cell used, recurses on four neighbors for `word[i+1]`, then restores the cell.

**Time:** `O(rows×cols×4^L)`. **Space:** `O(L)` recursion.

In [ ]:
def exist(board, word):
    rows, cols = len(board), len(board[0])
    def dfs(r, c, i):                      # can we match word[i:] starting at cell (r, c)?
        if i == len(word):
            return True                    # matched every letter
        if r < 0 or c < 0 or r >= rows or c >= cols or board[r][c] != word[i]:
            return False                   # off the board, or the letter doesn't match
        tmp = board[r][c]; board[r][c] = "#"   # mark this cell used (can't reuse within a word)
        found = (dfs(r+1,c,i+1) or dfs(r-1,c,i+1) or
                 dfs(r,c+1,i+1) or dfs(r,c-1,i+1))   # try all 4 neighbors for the next letter
        board[r][c] = tmp                  # restore the cell (backtrack)
        return found
    return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))   # try every start cell

In [ ]:
# Correctness check
board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]]
tests = [("ABCCED",True),("SEE",True),("ABCB",False)]
for word, exp in tests:
    got = exist([row[:] for row in board], word)
    print(f"{word} -> {got}")
    assert got == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)`       | ≈ **2×** |
| `O(n log n)` | ≈ **2×** (slightly more) |
| `O(n²)`     | ≈ **4×** |

*(Word Search is exponential in the word length; we grow the grid with a word that isn't present so the search fails after exploring.)*

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    board = [['A'] * n for _ in range(n)]   # all 'A'
    word = 'A' * 3 + 'B'                    # fails at the last letter after exploring
    return (board, word)
solutions = {
    "backtracking DFS": exist,
}
sizes = [20, 30, 40, 50]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Grid backtracking:** mark a cell used, explore neighbors, restore — the maze/word-path template.
- **Prune early:** bail the moment the current letter doesn't match.
- **Signal:** "find a path/word in a grid", "connected sequence of cells".
- **Related problems:** Word Search II (trie), Number of Islands, Robot Room Cleaner.
- **Common pitfalls:** (1) not restoring a cell (breaks other paths); (2) reusing a cell within one word.